# Simulated tutoring dialogue generator

Generates synthetic student-tutor dialogues for a simultaneous-equations problem, using a pipeline adapted from BEAGLE (Wang et al., 2026):

1. **Plan** the correct next step (`plan_correct_working`).
2. **Identify** which knowledge component (KC) that step exercises (`identify_kc`).
3. Consult the student's **BKT mastery** of that KC (`bkt.py`) to decide whether to inject a **misconception** (`generate_wrong_working`), gated by a Bayesian Knowledge Tracing model.
4. Voice the resulting working as the **student's** own turn, in a style set by the current **metacognitive state** (PLANNING / ENACTING / MONITORING / REFLECTING) sampled by a semi-Markov controller (`controller.py`).
5. Generate the **tutor's** guiding response.
6. **Judge** the tutor's response against an eight-dimension pedagogical rubric (`tutor_evaluation.py`).

Each stage can be toggled off independently via the feature flags in the main loop cell, to support ablation runs. Every run is logged turn-by-turn to `logs/` as a standalone HTML file.

**Setup:** copy `.env.example` to `.env` and add your own `OPENAI_API_KEY` before running. See `requirements.txt` for dependencies.

## Setup
Imports, configuration, fixed prompts, and the KC/misconception tables used throughout the run.

In [1]:
import os
import re
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
from bkt import BKT, KCS  # KCS (the knowledge-component list) is defined once, in bkt.py
from controller import SemiMarkovController, get_profile
from tutor_evaluation import evaluate_tutor_response

# Loads OPENAI_API_KEY from a local .env file (see .env.example) rather
# than hardcoding it in the notebook.
load_dotenv()
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# --- Config ---
# The maths problem the simulated student works through, how many turns
# to run, and which behavioural profile ("low" or "high") to sample from.
# See controller.py for what "low" vs "high" actually changes.
problem = "Solve the simultaneous equations: 3x + y = 11, 2x − y = 4"
n_turns = 10
BEHAV_PROFILE = "low"  # "low" or "high", passed to BKT() and SemiMarkovController()

# Seeds numpy's global RNG, which is what bkt.py and controller.py both
# draw from (np.random.uniform/choice/gamma etc). This makes the BKT
# mastery sampling and the semi-Markov state/duration sampling repeatable
# across runs. It does NOT make the OpenAI completions deterministic -
# those depend on the model's own sampling, so re-running with the same
# SEED will still produce different dialogue text.
SEED = 42
np.random.seed(SEED)

# Single source of truth for which OpenAI model every LLM call below uses.
MODEL = "gpt-4o"

# --- Personas and instructions (fixed across all turns) ---
# Each of these system_* strings is a system prompt for one of the five
# LLM calls defined in the cells below (see the function of the same
# step number: plan_correct_working, plan_wrong_working_nokc, identify_kc,
# generate_wrong_working, produce_student_output, produce_tutor_response).

# Used when planning is skipped (use_planning=False): invents an incorrect
# step from scratch, or corrupts a given correct plan, without going
# through KC identification or the BKT-gated misconception injection below.
system_1a = '''
Produce incorrect working for the next step of this simultaneous equations problem.
If a correct plan is given, corrupt it with a genuine mathematical error. If not, decide the next step yourself and introduce the error there.

Output ONLY:
Incorrect working: [wrong working, showing process]
'''

# Used to generate the correct next step of working (LLM call 1).
planning_instructions = '''
Plan the next small step of the working out.
If the tutor's last response asked a specific question or pointed to a specific issue, your plan MUST address that exact point - do not switch topic or method until it's resolved.

Write it in this format:
Plan: [next step]
Workings: [working for this step, showing process]

Produce exactly one plan - no alternative attempts or corrections-of-corrections.
'''

# Used to classify which knowledge component (KC) a plan is exercising
# (LLM call 2), so the BKT model can look up/update mastery for that KC.
system_2 = '''
You are an expert maths teacher identifying which knowledge component (KC) a student's plan is attempting.
Read their plan and the KC list below, and choose the single KC that most specifically matches the mechanical step they are about to perform.
If the plan could match more than one KC, choose the most specific one, not a more general related one.

Output ONLY: KC#
'''

# Used to turn a correct plan into flawed working by applying a specific,
# pre-written misconception for the identified KC (LLM call 3 - EFI).
system_3 = '''
Generate incorrect working by applying the given misconception to the given correct plan for this step.

Output ONLY:
Incorrect working: [wrong working, showing process]
'''

# Used to voice the (possibly flawed) working as the student's own
# in-character response (LLM call 4). When use_controller=True this is
# extended each turn with the current metacognitive-state profile from
# controller.py.
system_4 = '''
# Role
You are a novice student learning to solve simultaneous equations, thinking out loud as you work.

# Voice
- Use simple, direct language.
- "I think..." not "I hypothesize..."
- "It says..." not "The error indicates..."
- "That's weird" not "This is unexpected behavior"
- You may be confused. You may not know the right answer. You learn by doing, not by analysing.

# Instructions
- The working shown to you below is YOUR OWN thinking, as if you just came up with it yourself right now. It is not something you were told, given, or shown - it is what you believe.
- Do not self-correct, and do not signal that anything might be wrong.
- Take the tutor's last response into account, as if you are replying to them directly.
- ONLY produce the single SMALL step described. Do not do any more work than that. 
- Reason out loud throughout the step.

# Never do this
Do not use phrases that reveal you were given the working, such as:
- "According to my plan..."
- "The working I have is..."
- "I was told to..."
- "Following the instructions..."
- "As shown..." / "As given..."
Instead, just say what you're doing directly, e.g. "Let's try adding these two equations..." or "So if I substitute this in..."

# Output format
Write only what the student would actually say and write - no meta-commentary about the task itself.
'''

# Used to generate the tutor's guiding response to the student (LLM call 5).
system_5 = '''
# Role
You are a patient, encouraging maths tutor helping a student solve simultaneous equations.

# Task
Read the student's most recent working carefully, check the have attempted the correct step, it is mathematically correct, and respond with exactly one short guiding question or hint that moves them forward. Do not solve the step for them.

# Instructions
- Decide what the correct step should be, do the maths for the step yourself first, explicitly, before deciding whether the student is right or wrong.
- Before responding, check the student's working line by line for arithmetic and algebraic errors. Do not assume it is correct just because it looks confident.
- If the student has made an error, do not correct it directly. Ask a question that helps them notice it themselves (e.g. "What do you get if you substitute that back into the first equation?").
- If the student's working is correct, praise them, and ask a question that guides them to the next step, rather than stating what the next step is.
- Never give the final answer or complete the working for them unless they get the same step wrong 3 times. 
- Never introduce new mathematical content, methods, or numbers that the student has not already used - your job is to question and guide, not to teach a new technique.
- Base your response only on what the student actually wrote. Do not refer to steps they have not taken or assume progress they have not made.

# Output format
- Maximum three sentences.
- Ask exactly one question. Do not ask multiple questions in the same turn.
- Do not repeat back the student's working before responding - respond directly.
'''

# --- Misconceptions ---
# One pre-written misconception per KC (imported above from bkt.py), used
# by LLM call 3 to corrupt a correct plan in a specific, grounded way
# rather than an arbitrary/unconstrained error.
MISCONCEPTIONS = {
    "KC1":  "Collecting like terms: combining unlike terms by treating any two terms with letters as combinable",
    "KC2":  "Solving a one-step equation: applying the inverse operation to one side of the equation only",
    "KC3":  "Solving a multi-step equation: distributing a coefficient to the first term in a bracket only, not all terms",
    "KC4":  "Rearranging a formula: rearranging without changing the sign of the moved term",
    "KC5":  "Solving the resulting single-variable equation: making an arithmetic error at the final division step",
    "KC6":  "Back-substituting: stopping after finding the first variable and treating the solution as complete",
    "KC7":  "Checking the solution: checking in only one of the two original equations",
    "KC8":  "Recognising whether scaling is needed: attempting to add or subtract the equations immediately without checking whether coefficients match",
    "KC9":  "Scaling equations: multiplying only the left-hand side of the equation and not the right-hand side",
    "KC10": "Adding or subtracting to eliminate: adding the equations when they should be subtracted, or vice versa",
    "KC11": "Rearranging for substitution: rearranging without changing the sign of the moved term",
    "KC12": "Substituting into the second equation: substituting back into the same equation it came from rather than the other equation",
    "KC13": "Expanding after substitution: distributing a multiplier to the variable term only and not the constant inside the bracket",
}

# --- Running state ---
# `history` is the single source of truth for "what's happened so far"
# and is threaded into every LLM call below so each one has conversational
# context. It is appended to once per turn, after the tutor has replied.
# Full turn-by-turn history, shared by every LLM call that needs conversational context.
# Each entry: {"turn": int, "student": str, "tutor": str}
history = []

def format_history(history, max_turns=None):
    """Render the running history as a plain-text transcript for prompt context.
    max_turns=None includes everything; pass an int to keep only the most recent N turns
    if you ever need to cap token usage on very long runs."""
    if not history:
        return "No previous turns yet - this is the first turn of the conversation."
    turns = history if max_turns is None else history[-max_turns:]
    lines = []
    for h in turns:
        lines.append(f"Turn {h['turn']} - Student: {h['student']}")
        lines.append(f"Turn {h['turn']} - Tutor: {h['tutor']}")
    return "\n".join(lines)

# How many of the most recent turns to include as context in every LLM call.
# Keeps token usage bounded regardless of how many turns the run has - the
# looping problem is mostly fixed by recent context, not the entire history.
HISTORY_WINDOW = 3


## Pipeline functions
One function per LLM call in the pipeline described above.

In [2]:
# --- LLM call 1: Planning (correct working) ---

def plan_correct_working(problem, planning_instructions, planning_system, history):
    """Asks the model to work out the next correct step of the solution.

    Note planning_system is actually the student's system prompt
    (system_4, extended with the current metacognitive-state profile) -
    the plan is written in the student's own voice/state so that later
    steps can present it as something the student came up with.
    """
    user_1 = f'''
    Problem: {problem}
    {planning_instructions}
    Full conversation history so far:
    {format_history(history, max_turns=HISTORY_WINDOW)}
    '''

    response = client.chat.completions.create(
        model=MODEL, max_tokens=512,
        messages=[
            {"role": "system", "content": planning_system},
            {"role": "user", "content": user_1},
        ],
    )
    return response.choices[0].message.content

def plan_wrong_working_nokc(problem, history, system_1a, correct_plan=None):
    """Fallback path used when use_efi=False: produces incorrect working
    directly (by corrupting correct_plan if given, or inventing a step),
    skipping KC identification and the BKT-gated misconception lookup.
    """
    plan_block = (
        f"Correct plan for this step: {correct_plan}"
        if correct_plan
        else "No correct plan has been given — decide the next step yourself."
    )
    user_1a = f'''
    Problem: {problem}
    {plan_block}
    Full conversation history so far:
    {format_history(history, max_turns=HISTORY_WINDOW)}
    '''

    response = client.chat.completions.create(
        model=MODEL, max_tokens=512,
        messages=[
            {"role": "system", "content": system_1a},
            {"role": "user", "content": user_1a},
        ],
    )
    return response.choices[0].message.content



In [3]:
# --- LLM call 2: KC identification ---

def identify_kc(plan, KCS, kc_system):
    """Asks the model which KC (e.g. "KC5") the given plan is exercising.

    Falls back to KC1 if the model's reply doesn't contain a recognisable
    "KC<number>" pattern, so a malformed response can't crash the run.
    """
    user_2 = f'''
    Knowledge components: {KCS}
    Plan: {plan}
    '''

    response = client.chat.completions.create(
        model=MODEL, max_tokens=512,
        messages=[
            {"role": "system", "content": kc_system},
            {"role": "user", "content": user_2},
        ],
    )
    kc_result = response.choices[0].message.content

    match = re.search(r"KC\d+", kc_result)
    return match.group(0) if match else "KC1"

In [4]:
# --- LLM call 3: EFI (generate wrong working) ---
# EFI = Erroneous/Flawed working Injection: turns a correct plan into
# working that fails in one specific, pre-written way (see MISCONCEPTIONS
# above), rather than an unconstrained/random mistake.

def generate_wrong_working(plan, misconception, efi_system):
    """Applies `misconception` to the correct `plan` to produce flawed
    working, grounded in a specific KC rather than an arbitrary error.
    """
    user_3 = f'''
    Correct working: {plan}
    Misconception: {misconception}
    '''

    response = client.chat.completions.create(
        model=MODEL, max_tokens=512,
        messages=[
            {"role": "system", "content": efi_system},
            {"role": "user", "content": user_3},
        ],
    )
    return response.choices[0].message.content

In [5]:
# --- LLM call 4: Student output ---

def produce_student_output(problem, working, student_system, history):
    """Voices `working` (correct or flawed) as the student's own turn,
    in-character per student_system (system_4 + current metacognitive
    state), continuing naturally from `history`.
    """
    user_4 = f'''
    Problem: {problem}
    Working to produce: {working}
    Full conversation history so far (everything you and the tutor have said, in order):
    {format_history(history, max_turns=HISTORY_WINDOW)}
    '''

    response = client.chat.completions.create(
        model=MODEL, max_tokens=512,
        messages=[
            {"role": "system", "content": student_system},
            {"role": "user", "content": user_4},
        ],
    )
    return response.choices[0].message.content


In [6]:
# --- LLM call 5: Tutor ---

def produce_tutor_response(student_output, history, tutor_system, problem):
    """Generates the tutor's one-question guiding reply to the student's
    latest turn, per the rules in system_5 (never solve it for them, etc).
    """
    user_5 = f'''
    Problem: {problem}
    Student's current response (the one you are replying to): {student_output}
    Full conversation history so far (everything the student and you have said, in order):
    {format_history(history, max_turns=HISTORY_WINDOW)}
    '''

    response = client.chat.completions.create(
        model=MODEL, max_tokens=512,
        messages=[
            {"role": "system", "content": tutor_system},
            {"role": "user", "content": user_5},
        ],
    )
    return response.choices[0].message.content


## Main simulation loop
Runs `n_turns` of the pipeline end to end, applying the feature flags below, and builds an HTML transcript.

In [7]:
# --- Feature flags ---
# Toggle these to run ablations: each one switches off one stage of the
# pipeline while leaving the rest intact, so a run can isolate the effect
# of that stage. The output HTML/log filename records which were on
# (see the "suffix" logic in the next cell).
use_planning: bool = True    # False: skip planning, invent a wrong step directly (LLM call 1a only)
use_efi: bool = True  # False bypasses KC identification + EFI, student gets correct working
use_bkt_gate: bool = True    # False: always inject a misconception, ignoring BKT mastery
use_controller: bool = True  # False: no metacognitive-state persona, and BKT never updates
tutor_eval: bool = True  # False skips the tutor_evaluation.py judge call

bkt = BKT(BEHAV_PROFILE)
controller = SemiMarkovController(BEHAV_PROFILE)

logs = []       # structured record of every turn — one dict per turn
all_html = ""   # accumulated two-column display

for turn in range(n_turns):

    # --- Step 1: which metacognitive state is the student in this turn? ---
    # (only meaningful if use_controller=True; drives the student's voice below)
    meta_state = controller.get_state() if use_controller else None
    student_system_this_turn = (
        system_4 + "\n\n# Current metacognitive state\n" + get_profile(meta_state)
        if use_controller else system_4
    )

    # --- Steps 2-3: work out what the student will actually attempt this turn ---
    if use_planning:
        correct_plan = plan_correct_working(problem, planning_instructions, student_system_this_turn, history)

        if use_efi:
            # Identify the KC being exercised, look up the student's current
            # mastery of it, and use the BKT gate to decide whether to inject
            # a misconception (or always inject, if the gate is disabled).
            kc_key = identify_kc(correct_plan, KCS, system_2)
            kc_label = f"{kc_key} — {KCS.get(kc_key, '?')}"
            mastery_level = bkt.get_mastery_level(kc_key)
            inject = bkt.should_inject_misconception(kc_key) if use_bkt_gate else True

            if inject:
                misconception = MISCONCEPTIONS.get(kc_key, "Unknown misconception")
                working = generate_wrong_working(correct_plan, misconception, system_3)
            else:
                misconception = "N/A (gated out — mastery too high)"
                working = correct_plan

            bkt_updated = use_controller and meta_state in ("MONITORING", "REFLECTING")
            if bkt_updated:
                bkt.update(kc_key)
        else:
            kc_label = "N/A (ungrounded error, based on correct plan)"
            mastery_level = "N/A"
            inject = None
            bkt_updated = False
            misconception = "N/A (ungrounded error, based on correct plan)"
            working = plan_wrong_working_nokc(problem, history, system_1a, correct_plan=correct_plan)
    else:
        kc_label = "N/A (ungrounded error, no correct plan)"
        mastery_level = "N/A"
        inject = None
        bkt_updated = False
        misconception = "N/A (ungrounded error, no correct plan)"
        working = plan_wrong_working_nokc(problem, history, system_1a, correct_plan=None)

    # --- Step 4: student voices the working out loud (correct or flawed) ---
    student_output = produce_student_output(problem, working, student_system_this_turn, history)

    # --- Step 5: tutor reads the student's turn and responds with a hint/question ---
    tutor_output = produce_tutor_response(student_output, history, system_5, problem)

    # --- Step 6: Tutor evaluation (judged against the history BEFORE this turn is appended) ---
    if tutor_eval:
        tutor_eval_result = evaluate_tutor_response(format_history(history, max_turns=HISTORY_WINDOW), tutor_output)
    else:
        tutor_eval_result = None

    # --- Update running history (used by every call above on the NEXT turn) ---
    history.append({"turn": turn + 1, "student": student_output, "tutor": tutor_output})

    if use_controller:
        controller.step()

    # --- Log this turn ---
    turn_log = {
        "turn": turn + 1,
        "use_planning": use_planning,
        "use_efi": use_efi,
        "use_bkt_gate": use_bkt_gate,
        "use_controller": use_controller,
        "tutor_eval": tutor_eval,
        "correct_plan": correct_plan if use_planning else None,
        "kc_label": kc_label,
        "mastery_level": mastery_level,
        "injected": inject,
        "meta_state": meta_state,
        "bkt_updated": bkt_updated,
        "misconception": misconception,
        "working": working,
        "student_output": student_output,
        "tutor_output": tutor_output,
        "tutor_eval_result": tutor_eval_result,
    }
    logs.append(turn_log)

    left_col = f"""
        <div>
        <h4>Turn {turn + 1}</h4>
        <p><b>🎓 Student:</b> {student_output}</p>
        <p><b>👩‍🏫 Tutor:</b> {tutor_output}</p>
        </div>
    """

    correct_plan_display = correct_plan if use_planning else "<i>N/A — planning bypassed</i>"

    if tutor_eval_result:
        tutor_eval_summary = " | ".join(f"{k}: {v['label']}" for k, v in tutor_eval_result.items())
    else:
        tutor_eval_summary = "N/A (tutor_eval off)"

    right_col = f"""
        <div>
        <p><b>Config:</b> use_planning={use_planning}, use_efi={use_efi}</p>
        <p><b>Mastery:</b> {mastery_level} | <b>Injected:</b> {inject}</p>
        <p><b>Meta-state:</b> {meta_state} | <b>BKT updated:</b> {bkt_updated}</p>
        <p><b>Correct plan:</b> {correct_plan_display}</p>
        <p><b>KC:</b> {kc_label}</p>
        <p><b>Misconception:</b> {misconception}</p>
        <p><b>Working passed to student:</b> {working}</p>
        <p><b>Tutor eval:</b> {tutor_eval_summary}</p>
        </div>
    """

    all_html += f"""
        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px; border-top: 2px solid #ccc; padding-top: 12px; margin-top: 12px;">
            <div style="background: #f9f9f9; padding: 12px; border-radius: 8px;">{left_col}</div>
            <div style="background: #f0f4ff; padding: 12px; border-radius: 8px;">{right_col}</div>
        </div>
        """

# --- Outside the loop — runs once, after all turns are done ---
# from IPython.display import HTML
# display(HTML(all_html))

## Save transcript
Writes the accumulated HTML transcript to `logs/`.

In [8]:
# Writes a standalone HTML transcript of the run to logs/, named with a
# timestamp and a suffix that encodes which feature flags were on above
# (e.g. run_20260101_120000_p_efi_gate_ctrl.html), so runs stay distinguishable
# without needing to reopen the notebook.
import os
from datetime import datetime

os.makedirs("logs", exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

if use_planning and use_efi:
    suffix = "_p_efi"
elif use_planning and not use_efi:
    suffix = "_p"
else:
    suffix = ""

if use_efi and use_bkt_gate:
    suffix += "_gate"

if use_controller:
    suffix += "_ctrl"

log_filename = f"logs/run_{timestamp}{suffix}.html"

full_html = f"""
<html>
<head>
<meta charset="utf-8">
<title>Run {timestamp}</title>
<script>
MathJax = {{
  tex: {{ inlineMath: [['\\\\(', '\\\\)'], ['$', '$']], displayMath: [['\\\\[', '\\\\]']] }},
  svg: {{ fontCache: 'global' }}
}};
</script>
<script async src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-svg.js"></script>
</head>
<body style="font-family: sans-serif; max-width: 1100px; margin: 40px auto;">
<h2>Simulation run — {timestamp}</h2>
<p>Config: use_planning={use_planning}, use_efi={use_efi}, use_bkt_gate={use_bkt_gate}, use_controller={use_controller}, tutor_eval={tutor_eval}, meta_state={meta_state}, problem="{problem}", n_turns={n_turns}</p>
{all_html}
</body>
</html>
"""

with open(log_filename, "w", encoding="utf-8") as f:
    f.write(full_html)

print(f"Log saved to {log_filename}")

Log saved to logs/run_20260814_100452_p_efi_gate_ctrl.html
